<a href="https://colab.research.google.com/github/daniausman24-bot/ML_Internship_Track/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daniausman24-bot/ML_Internship_Track/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

I will prioritize content for review when it is old, has measurable search visibility, and has a weak average search position. The rule is directional decision-support, not a prediction of why a page declined.

Reason codes:

* `stale_visible_weak_position` — old page, meaningful impressions, and weak measured position.
* `stale_visible` — old page with meaningful impressions.
* `visible_weak_position` — meaningful impressions and weak measured position.
* `stale` — old page without the other two signals.
* `weak_position` — weak measured position with meaningful position data.
* `visible` — meaningful impressions without the other signals.
* `no_priority_signal` — none of the review conditions were met.

The score increases when more review signals are present. Pages with more signals are ranked earlier.


In [ ]:
# Baseline thresholds
STALE_DAYS = 180
MIN_IMPRESSIONS = 500
WEAK_POSITION = 10

print("Baseline thresholds")
print("Stale:", STALE_DAYS, "days")
print("Minimum impressions:", MIN_IMPRESSIONS)
print("Weak position:", WEAK_POSITION)

Baseline thresholds
Stale: 180 days
Minimum impressions: 500
Weak position: 10


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

import pandas as pd
import numpy as np
from pathlib import Path

# Load starter data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Recreate the label
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

# -----------------------------
# Baseline signals
# -----------------------------

df["stale_flag"] = (
    df["days_since_last_update"] >= STALE_DAYS
).astype(int)

df["visible_flag"] = (
    df["impressions_90d"] >= MIN_IMPRESSIONS
).astype(int)

# avg_position = 0 means no position data
df["weak_position_flag"] = (
    (df["avg_position"] > 0) &
    (df["avg_position"] >= WEAK_POSITION)
).astype(int)

# -----------------------------
# Transparent score
# -----------------------------

df["baseline_score"] = (
    df["stale_flag"]
    + df["visible_flag"]
    + df["weak_position_flag"]
)

# -----------------------------
# Reason codes
# -----------------------------

def make_reason(row):
    stale = row["stale_flag"]
    visible = row["visible_flag"]
    weak = row["weak_position_flag"]

    if stale and visible and weak:
        return "stale_visible_weak_position"
    elif stale and visible:
        return "stale_visible"
    elif visible and weak:
        return "visible_weak_position"
    elif stale:
        return "stale"
    elif weak:
        return "weak_position"
    elif visible:
        return "visible"
    else:
        return "no_priority_signal"

df["reason_code"] = df.apply(make_reason, axis=1)

# -----------------------------
# Rank
# -----------------------------

df = df.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

df["baseline_rank"] = np.arange(1, len(df) + 1)

# -----------------------------
# Write output
# -----------------------------

output_cols = [
    "baseline_rank",
    "content_id",
    "client_id",
    "baseline_score",
    "reason_code",
    "stale_flag",
    "visible_flag",
    "weak_position_flag",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "is_declining_label"
]

baseline_output = df[output_cols].copy()

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

baseline_output.to_csv(output_path, index=False)

print("Output written to:", output_path)
print("Rows:", len(baseline_output))
print("\nScore distribution:")
print(df["baseline_score"].value_counts().sort_index(ascending=False))

print("\nReason codes:")
print(df["reason_code"].value_counts())

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.
Output written to: work/outputs/baseline_action_score.csv
Rows: 30000

Score distribution:
baseline_score
3       14
2     9284
1    14252
0     6450
Name: count, dtype: int64

Reason codes:
reason_code
visible_weak_position          9232
visible                        7477
weak_position                  6667
no_priority_signal             6450
stale                           157
stale_visible_weak_position      14
stale_visible                     3
Name: count, dtype: int64


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Rank 1 — Action: Review/update.
Reason: stale_visible_weak_position.
Confidence: Moderate because all three rule signals are observed.
Could be wrong if: The page is intentionally old or the position measurement is not representative.

In [ ]:
top20 = df.head(20).copy()

print(
    top20[
        [
            "baseline_rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "impressions_90d",
            "days_since_last_update",
            "avg_position",
            "is_declining_label"
        ]
    ].to_string(index=False)
)

 baseline_rank           content_id  baseline_score                 reason_code  impressions_90d  days_since_last_update  avg_position  is_declining_label
             1 content_cf56e2e2e282               3 stale_visible_weak_position            61678                     194          19.7                   1
             2 content_7368877ea310               3 stale_visible_weak_position            59472                     194          24.8                   1
             3 content_1bfaa38ff26c               3 stale_visible_weak_position            25715                     194          22.2                   1
             4 content_0a91db491d14               3 stale_visible_weak_position            13299                     193          10.5                   1
             5 content_5feee3994adb               3 stale_visible_weak_position             7812                     194          39.0                   1
             6 content_c2d929d83eaa               3 stale_visible_weak

In [ ]:
k = 20

precision_at_20 = df.head(k)["is_declining_label"].mean()
base_rate = df["is_declining_label"].mean()

print("Precision@20:", precision_at_20)
print("Base rate:", base_rate)
print("Top-20 declining rows:", df.head(k)["is_declining_label"].sum())

Precision@20: 0.8
Base rate: 0.5420666666666667
Top-20 declining rows: 16


## 3. Top-20 review

The baseline ranked 20 content items for review. Precision@20 was 0.80, meaning 16 of the 20 selected items were observed as declining. The overall declining rate was 0.5421, so the Top-20 precision was about 25.8 percentage points higher than the base rate.

The first 14 items mostly received the strongest score because they were stale, visible, and had weak measured positions. Ranks 15–20 received a score of 2 because they were visible and had weak measured positions but were not yet stale.

| Rank | Action        | Reason code                   | Confidence note | What could make it wrong                                                  |
| ---- | ------------- | ----------------------------- | --------------- | ------------------------------------------------------------------------- |
| 1    | Review/update | `stale_visible_weak_position` | High            | The page may be intentionally old despite weak position.                  |
| 2    | Review/update | `stale_visible_weak_position` | High            | The page may not need an update if its topic is still current.            |
| 3    | Review/update | `stale_visible_weak_position` | High            | Weak position may be caused by competition rather than stale content.     |
| 4    | Review/update | `stale_visible_weak_position` | High            | The measured position may not represent every query.                      |
| 5    | Review/update | `stale_visible_weak_position` | High            | A high impression count does not guarantee that an update will help.      |
| 6    | Review/update | `stale_visible_weak_position` | High            | The page may be intentionally maintained without recent changes.          |
| 7    | Review/update | `stale_visible_weak_position` | High            | Weak ranking may have causes outside the content itself.                  |
| 8    | Review/update | `stale_visible_weak_position` | High            | Position alone does not prove content quality problems.                   |
| 9    | Review/update | `stale_visible_weak_position` | High            | Search demand or competition may have changed.                            |
| 10   | Review/update | `stale_visible_weak_position` | High            | The page may already be planned for another reason.                       |
| 11   | Review/update | `stale_visible_weak_position` | Moderate        | The rule selected it, but the observed label is not declining.            |
| 12   | Review/update | `stale_visible_weak_position` | High            | Low impressions make the action less certain than higher-volume pages.    |
| 13   | Review/update | `stale_visible_weak_position` | Moderate        | The page has fewer impressions and may have limited review value.         |
| 14   | Review/update | `stale_visible_weak_position` | Moderate        | The very weak position may reflect limited query coverage.                |
| 15   | Review/update | `visible_weak_position`       | Moderate        | It is not stale, so a content update may not be the right action.         |
| 16   | Review/update | `visible_weak_position`       | Moderate        | High impressions do not show that content changes are needed.             |
| 17   | Review/update | `visible_weak_position`       | Moderate        | The page is relatively recent, so staleness is not supporting the action. |
| 18   | Review/update | `visible_weak_position`       | Moderate        | The page is declining, but the rule does not identify the cause.          |
| 19   | Review/update | `visible_weak_position`       | Moderate        | The page is not stale, so ranking weakness may have another explanation.  |
| 20   | Review/update | `visible_weak_position`       | Moderate        | The page is not stale and may require investigation before updating.      |

The strongest evidence comes from the combination of visibility, age, and measured position. The score is still only a review priority and should not be treated as proof that updating a page will improve its performance.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Rank 11 is a clear weak pick. It received the maximum baseline score of 3 and the reason code stale_visible_weak_position, but its observed declining label is 0. This shows that the rule can prioritize content that is old, visible, and weakly positioned even when the measured trend is not declining.

The baseline should therefore be treated as a review queue rather than a prediction that a page is declining.

The leakage check confirms that the score uses only days_since_last_update, impressions_90d, and avg_position. The target field and the last/previous 30-day trend fields are not used to calculate the score. provider_used and model_used are also excluded.

The score therefore does not use the observed target or the future/label-defining trend fields. The result is directional decision-support, not a causal or predictive claim.

In [ ]:
# Show the weakest-scoring items within the top 20
print("Top-20 scores:")
print(df.head(20)["baseline_score"].value_counts().sort_index())

# Check for top-20 rows with no priority signal
weak_top20 = df.head(20)[
    df.head(20)["reason_code"] == "no_priority_signal"
]

print("\nTop-20 rows with no priority signal:")
print(
    weak_top20[
        ["baseline_rank", "content_id", "baseline_score", "reason_code"]
    ].to_string(index=False)
)

# -----------------------------
# Leakage check
# -----------------------------

future_or_label_cols = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

score_inputs = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position"
]

print("\nScore inputs:")
print(score_inputs)

print("\nLeakage columns excluded from score:")
print(future_or_label_cols)

# Verify none of the forbidden columns are used
leaked_inputs = [
    c for c in future_or_label_cols
    if c in score_inputs
]

print("\nForbidden columns used in score:", leaked_inputs)

assert len(leaked_inputs) == 0

# Product/model flag check
product_flag_cols = [
    "provider_used",
    "model_used"
]

print("\nProduct/model columns:")
print(product_flag_cols)

print("\nProduct/model columns used in score:")
print([
    c for c in product_flag_cols
    if c in score_inputs
])

print("\nLeakage check passed.")

Top-20 scores:
baseline_score
2     6
3    14
Name: count, dtype: int64

Top-20 rows with no priority signal:
Empty DataFrame
Columns: [baseline_rank, content_id, baseline_score, reason_code]
Index: []

Score inputs:
['days_since_last_update', 'impressions_90d', 'avg_position']

Leakage columns excluded from score:
['trend_direction', 'trend_pct', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

Forbidden columns used in score: []

Product/model columns:
['provider_used', 'model_used']

Product/model columns used in score:
[]

Leakage check passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.